In [ ]:
from torch import nn
import torch

class PatchEmbedding(nn.Module):
    def __init__(self, input_dim=3, output_dim=768, patch_size=16, img_size=224):
        super().__init__()
        self.conv = nn.Conv2d(input_dim, output_dim, kernel_size=patch_size, stride=patch_size)
        # Initial dimensions (C, H, W) eg. (3, 224, 224)
        # Output dimensions (output_dim, (H_in - kernel_size) / stride + 1, (W_in - kernel_size) / stride + 1)
        # eg. (768, (224 - 16) / 16 + 1 = 14, (224 - 16) / 16 + 1 = 14)
        # (768, 14, 14)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, output_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, (img_size // patch_size) ** 2 + 1, output_dim))

    def forward(self, x):
        # [B, C, H, W]
        x = self.conv(x)
        # [B, output_dim, H_out, W_out]
        # To flatten the spatial dimensions into a single dimension
        x = x.flatten(2)
        # [B, output_dim, H_out * W_out]
        x = x.transpose(1, 2)
        # [B, H_out * W_out, output_dim]
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), x], dim=1)
        # [B, H_out * W_out + 1, output_dim]
        x = x + self.pos_embed
        return x

class Normalization(nn.Module):
    def __init__(self, eps=1e-6, dim=768):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        # Formula: y = gamma * (x - mean) / sqrt(variance + epsilon) + beta

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x = self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0, "Dimension must be divisible by number of heads"
        self.num_heads = num_heads
        self.w_k = nn.Linear(dim, dim)
        self.w_q = nn.Linear(dim, dim)
        self.w_v = nn.Linear(dim, dim)
        self.w_o = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(q,k,v, dropout=None):
        k = k.transpose(-2, -1)
        # [B, num_heads, seq_len, head_dim] -> [B, num_heads, head_dim, seq_len]
        attention_scores = torch.softmax((q @ k)/torch.sqrt(torch.tensor(q.size(-1), dtype=torch.float32)), dim=-1)
        if dropout is not None:
          attention_scores = dropout(attention_scores)
        return attention_scores @ v

    def forward(self, x):
        k = self.w_k(x)
        q = self.w_q(x)
        v = self.w_v(x)
        k = k.view(k.size(0), k.size(1), self.num_heads, -1).transpose(1,2)
        q = q.view(q.size(0), q.size(1), self.num_heads, -1).transpose(1,2)
        v = v.view(v.size(0), v.size(1), self.num_heads, -1).transpose(1,2)
        attention = self.attention(q,k,v, self.dropout)
        x = attention.transpose(1,2).contiguous().view(x.size(0), x.size(1), -1)
        return self.w_o(x)

class MLP(nn.Module):
    def __init__(self, dim=768, hidden_dim=3072, dropout=0.1):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        return self.sequential(x)

class ImageEncoder(nn.Module):
    def __init__(self, num_layers=12, input_dim=3, output_dim=768, patch_size=16, img_size=224, num_heads=8, mlp_hidden_dim=3072, dropout=0.1):
        super().__init__()
        self.patch_embedding = PatchEmbedding(input_dim, output_dim, patch_size, img_size)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "norm1": Normalization(dim=output_dim),
                "attn": MultiHeadAttention(dim=output_dim, num_heads=num_heads, dropout=dropout),
                "norm2": Normalization(dim=output_dim),
                "mlp": MLP(dim=output_dim, hidden_dim=mlp_hidden_dim, dropout=dropout)
            })
            for _ in range(num_layers)
        ])

    def forward(self, x):
        x = self.patch_embedding(x)
        for layer in self.layers:
            x = x + layer["attn"](layer["norm1"](x))
            x = x + layer["mlp"](layer["norm2"](x))
        return x[:, 0]